# CARMS — Phase 1 Exploration
Visual inspection of downloaded data, features, charts, news and macro.

In [1]:
# ── Fix working directory to project root ────────────────────
import os, sys
from pathlib import Path

# Walk up until we find main.py (= project root)
root = Path(os.getcwd())
for _ in range(4):
    if (root / 'main.py').exists():
        break
    root = root.parent

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(f'Project root : {root}')
print(f'data/raw     : {(root / "data/raw").exists()}')
print(f'data/processed: {(root / "data/processed").exists()}')

Project root : C:\Users\Mugithi\Documents\carms
data/raw     : True
data/processed: True


In [2]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#f8f9fa',
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'font.size':        10,
})

from src.utils.logger import load_config
from src.utils.validator import validate_phase1, print_feature_summary
from src.ingestion.downloader import load_all_assets
from src.ingestion.news_fetcher import load_news, load_macro
from src.features.indicators import load_features, get_feature_columns
from src.features.chart_generator import load_chart_metadata

config = load_config('configs/config.yaml')
print('✓ Config loaded')

✓ Config loaded


## 1 — Validation report

In [3]:
ok = validate_phase1(config)
print('\nPhase 1 status:', '✅ COMPLETE' if ok else '⚠️  INCOMPLETE')


────────────────────────────────────────────────────────────
  CARMS Phase 1 — Validation Report
────────────────────────────────────────────────────────────

  Raw OHLCV Data
    ✓ EURUSD=X         daily     1,564 rows
    ✓ EURUSD=X         hourly   17,241 rows
    ✓ KES=X            daily     1,564 rows
    ✓ KES=X            hourly    7,442 rows
    ✓ BTC-USD          daily     2,191 rows
    ✓ BTC-USD          hourly   17,330 rows
    ✓ ETH-USD          daily     2,191 rows
    ✓ ETH-USD          hourly   17,330 rows
    ✓ GC=F             daily     1,509 rows
    ✓ GC=F             hourly   13,706 rows

  Processed Features
    ✓ EURUSD=X          1,515 rows × 29 features
    ✓ KES=X             1,515 rows × 29 features
    ✓ BTC-USD           2,142 rows × 29 features
    ✓ ETH-USD           2,142 rows × 29 features
    ✓ GC=F              1,460 rows × 29 features

  Chart Images (CNN Input)
    ✓ EURUSD=X          1,490 imgs     100 metadata  up-label: 46.0%
    ✓ KES=X        

## 2 — Raw OHLCV price history

In [4]:
assets = load_all_assets(config, interval='daily')
print(f'Loaded {len(assets)} assets:')
for sym, df in assets.items():
    print(f'  {sym:<14} {len(df):>6,} rows   {df.index[0].date()} → {df.index[-1].date()}')

Loaded 5 assets:
  EURUSD=X        1,564 rows   2019-01-01 → 2024-12-30
  KES=X           1,564 rows   2019-01-01 → 2024-12-30
  BTC-USD         2,191 rows   2019-01-01 → 2024-12-30
  ETH-USD         2,191 rows   2019-01-01 → 2024-12-30
  GC=F            1,509 rows   2019-01-02 → 2024-12-30


In [5]:
COLOURS = {
    'EURUSD=X': '#378ADD',
    'KES=X':    '#1D9E75',
    'BTC-USD':  '#F7931A',
    'ETH-USD':  '#627EEA',
    'GC=F':     '#D4AF37',
}
LABELS = {
    'EURUSD=X': 'EUR/USD',
    'KES=X':    'USD/KES',
    'BTC-USD':  'Bitcoin',
    'ETH-USD':  'Ethereum',
    'GC=F':     'Gold',
}

n = len(assets)
fig, axes = plt.subplots(n, 1, figsize=(14, 2.8 * n), sharex=False)
if n == 1: axes = [axes]

for ax, (sym, df) in zip(axes, assets.items()):
    col   = COLOURS.get(sym, '#888')
    label = LABELS.get(sym, sym)
    # Normalised close (index = 100 at start)
    norm  = df['close'] / df['close'].iloc[0] * 100
    ax.plot(df.index, norm, color=col, linewidth=1.2, label=label)
    ax.fill_between(df.index, 100, norm, alpha=0.08, color=col)
    ax.axhline(100, color='gray', linestyle='--', linewidth=0.5, alpha=0.5)
    ax.set_ylabel('Indexed (100)', fontsize=9)
    ax.set_title(f'{label}  ({sym})', fontsize=10, fontweight='bold', loc='left')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    chg = (df['close'].iloc[-1] / df['close'].iloc[0] - 1) * 100
    ax.text(0.98, 0.88, f'{chg:+.1f}% total', transform=ax.transAxes,
            ha='right', fontsize=9, color=col, fontweight='bold')

fig.suptitle('CARMS — Asset Price History (indexed to 100)', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
Path('logs').mkdir(exist_ok=True)
plt.savefig('logs/01_price_history.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved → logs/01_price_history.png')

Saved → logs/01_price_history.png


In [6]:
# Returns & volatility overview

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 3a — Annualised volatility bar chart
vols, labels, colors = [], [], []
for sym, df in assets.items():
    ret  = df['close'].pct_change().dropna()
    vol  = ret.std() * np.sqrt(252) * 100
    vols.append(vol)
    labels.append(LABELS.get(sym, sym))
    colors.append(COLOURS.get(sym, '#888'))

bars = axes[0].bar(labels, vols, color=colors, edgecolor='white', linewidth=0.5)
axes[0].set_title('Annualised Volatility (%)', fontweight='bold')
axes[0].set_ylabel('%')
for bar, v in zip(bars, vols):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{v:.1f}%', ha='center', va='bottom', fontsize=8)

# 3b — Correlation matrix heatmap
ret_df = pd.DataFrame({
    LABELS.get(s, s): df['close'].pct_change()
    for s, df in assets.items()
}).dropna()
corr = ret_df.corr()
im = axes[1].imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
axes[1].set_xticks(range(len(corr)))
axes[1].set_yticks(range(len(corr)))
axes[1].set_xticklabels(corr.columns, rotation=40, ha='right', fontsize=8)
axes[1].set_yticklabels(corr.columns, fontsize=8)
for i in range(len(corr)):
    for j in range(len(corr)):
        axes[1].text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center',
                     fontsize=7, color='black' if abs(corr.iloc[i,j]) < 0.6 else 'white')
plt.colorbar(im, ax=axes[1], fraction=0.04)
axes[1].set_title('Return Correlation Matrix', fontweight='bold')

# 3c — Rolling 30-day volatility
for sym, df in assets.items():
    roll_vol = df['close'].pct_change().rolling(30).std() * np.sqrt(252) * 100
    axes[2].plot(df.index, roll_vol, label=LABELS.get(sym, sym),
                 color=COLOURS.get(sym, '#888'), linewidth=0.8)
axes[2].set_title('Rolling 30-day Volatility (%)', fontweight='bold')
axes[2].set_ylabel('%')
axes[2].legend(fontsize=7)
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.suptitle('Returns & Volatility Analysis', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('logs/02_volatility.png', dpi=130, bbox_inches='tight')
plt.show()

In [7]:
# Technical features deep-dive
# Choose symbol to inspect
SYMBOL = 'BTC-USD'

feat = load_features(SYMBOL)
if feat is None:
    print(f'No features for {SYMBOL} — run python main.py --phase 1')
else:
    print_feature_summary(SYMBOL)
    feat_cols = get_feature_columns(feat)
    print(f'\nAll features ({len(feat_cols)}): {feat_cols}')


BTC-USD Feature Summary
  Date range : 2019-02-19 → 2024-12-30
  Rows       : 2,142
  Features   : 23

  Feature                      Mean        Std        Min        Max
  ────────────────────── ────────── ────────── ────────── ──────────
  return_1d                  0.0021     0.0341    -0.3717     0.1875
  log_return                 0.0015     0.0345    -0.4647     0.1718
  volatility_20              0.4954     0.2297     0.0924     1.9769
  range_hl                   0.0434     0.0339     0.0036     0.6174
  rsi_14                    53.6806    13.9804    14.0805    90.7194
  macd                     305.2130  1479.8097 -5053.2741  7049.2176
  macd_signal                1.0178   403.1844 -1694.5182  1882.8556
  macd_hist                304.1952  1409.5144 -4438.9991  6427.7841
  bb_upper               28268.7561 19459.4257  3283.8160 93601.5473
  bb_mid                 31696.3675 21456.0369  3601.7945 99799.2551
  bb_lower               35123.9790 23683.2826  3899.5547 106599.086

In [8]:
if feat is not None:
    fig = plt.figure(figsize=(15, 10))
    gs  = gridspec.GridSpec(4, 1, hspace=0.05)

    # Panel 1 — Close price + EMAs
    ax1 = fig.add_subplot(gs[0])
    ax1.plot(feat.index, feat['close'],  color='#F7931A', linewidth=1,   label='Close', zorder=3)
    ax1.plot(feat.index, feat['ema_20'], color='#378ADD', linewidth=0.8, label='EMA-20', linestyle='--')
    ax1.plot(feat.index, feat['ema_50'], color='#D85A30', linewidth=0.8, label='EMA-50', linestyle='--')
    ax1.set_ylabel('Price')
    ax1.legend(loc='upper left', fontsize=8)
    ax1.set_title(f'{SYMBOL} — Technical Indicators', fontweight='bold', fontsize=11)
    ax1.set_xticklabels([])

    # Panel 2 — RSI
    ax2 = fig.add_subplot(gs[1], sharex=ax1)
    ax2.plot(feat.index, feat['rsi_14'], color='#7F77DD', linewidth=0.8)
    ax2.axhline(70, color='red',   linestyle='--', linewidth=0.6, alpha=0.7)
    ax2.axhline(30, color='green', linestyle='--', linewidth=0.6, alpha=0.7)
    ax2.axhline(50, color='gray',  linestyle=':',  linewidth=0.5, alpha=0.5)
    ax2.fill_between(feat.index, feat['rsi_14'], 70,
                     where=feat['rsi_14'] >= 70, alpha=0.15, color='red')
    ax2.fill_between(feat.index, feat['rsi_14'], 30,
                     where=feat['rsi_14'] <= 30, alpha=0.15, color='green')
    ax2.set_ylabel('RSI-14')
    ax2.set_ylim(0, 100)
    ax2.set_xticklabels([])

    # Panel 3 — MACD
    ax3 = fig.add_subplot(gs[2], sharex=ax1)
    ax3.plot(feat.index, feat['macd'],        color='#378ADD', linewidth=0.8, label='MACD')
    ax3.plot(feat.index, feat['macd_signal'], color='#D85A30', linewidth=0.8, label='Signal')
    ax3.bar(feat.index, feat['macd_hist'],
            color=feat['macd_hist'].apply(lambda x: '#1D9E75' if x >= 0 else '#E24B4A'),
            width=1, alpha=0.6, label='Histogram')
    ax3.axhline(0, color='gray', linewidth=0.5)
    ax3.set_ylabel('MACD')
    ax3.legend(loc='upper left', fontsize=7)
    ax3.set_xticklabels([])

    # Panel 4 — Bollinger Band width (volatility proxy)
    ax4 = fig.add_subplot(gs[3], sharex=ax1)
    ax4.plot(feat.index, feat['bb_width'],    color='#BA7517', linewidth=0.8, label='BB width')
    ax4.plot(feat.index, feat['volatility_20'], color='#993C1D', linewidth=0.8,
             linestyle='--', label='Realised vol (20d)')
    ax4.set_ylabel('Volatility')
    ax4.legend(loc='upper left', fontsize=7)
    ax4.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

    plt.savefig(f'logs/03_{SYMBOL.replace("-","_")}_indicators.png', dpi=130, bbox_inches='tight')
    plt.show()
    print(f'Saved → logs/03_{SYMBOL}_indicators.png')

Saved → logs/03_BTC-USD_indicators.png


In [9]:
# Feature correlation heatmap

if feat is not None:
    feat_cols = get_feature_columns(feat)
    corr = feat[feat_cols].corr()

    fig, ax = plt.subplots(figsize=(13, 10))
    im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    ax.set_xticks(range(len(corr)))
    ax.set_yticks(range(len(corr)))
    ax.set_xticklabels(corr.columns, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(corr.columns, fontsize=8)
    for i in range(len(corr)):
        for j in range(len(corr)):
            v = corr.iloc[i, j]
            ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                    fontsize=6, color='white' if abs(v) > 0.6 else 'black')
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.01)
    ax.set_title(f'{SYMBOL} — Feature Correlation Matrix ({len(feat_cols)} features)',
                 fontweight='bold', fontsize=11)
    plt.tight_layout()
    plt.savefig('logs/04_feature_correlation.png', dpi=130, bbox_inches='tight')
    plt.show()

In [10]:
# Chart images preview (CNN input)

from PIL import Image
from pathlib import Path

sym_safe = SYMBOL.replace('-', '_').replace('=', '_')
chart_dir = Path('data/charts') / sym_safe
meta      = load_chart_metadata(SYMBOL)

if chart_dir.exists() and meta is not None and len(meta) > 0:
    img_files = sorted(chart_dir.glob('*.png'))[:16]
    n_show    = len(img_files)
    cols      = 8
    rows      = (n_show + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.4, rows * 1.6))
    axes = axes.flat if rows > 1 else [axes] if cols == 1 else axes.flat

    for ax, img_path in zip(axes, img_files):
        try:
            img = Image.open(img_path)
            ax.imshow(img, cmap='gray' if img.mode == 'L' else None, aspect='auto')
            # Find label from metadata
            date_str = img_path.stem[:8]
            row = meta[meta['date'].astype(str).str[:10].str.replace('-','') == date_str]
            if not row.empty:
                lbl = row['label_1d'].values[0]
                ret = row['return_1d'].values[0]
                ax.set_title(f"{'▲' if lbl else '▼'} {ret*100:+.1f}%",
                             fontsize=7,
                             color='#1D9E75' if lbl else '#E24B4A')
        except Exception:
            ax.text(0.5, 0.5, 'err', ha='center', va='center', fontsize=6)
        ax.axis('off')

    # Hide unused axes
    for ax in list(axes)[n_show:]:
        ax.set_visible(False)

    plt.suptitle(f'{SYMBOL} — Sample Candlestick Chart Images\n'
                 f'(▲ = next-day up, ▼ = next-day down, % = next-day return)',
                 fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.savefig('logs/05_chart_images.png', dpi=130, bbox_inches='tight')
    plt.show()
    print(f'Label balance: {meta["label_1d"].mean():.1%} up / {1-meta["label_1d"].mean():.1%} down')
else:
    print(f'No chart images found for {SYMBOL}.')
    print(f'Expected: {chart_dir}')

Label balance: 62.0% up / 38.0% down


In [11]:
# News & macro data

news = load_news()
if news is not None and not news.empty:
    print(f'News corpus: {len(news):,} articles')
    print(f'Date range : {news["published_at"].min()} → {news["published_at"].max()}')
    print(f'\nArticles per asset:')
    print(news['symbol'].value_counts().to_string())
    print(f'\nSample headlines:')
    for _, row in news.sample(min(5, len(news))).iterrows():
        print(f"  [{row['symbol']}] {row['title'][:80]}")
else:
    print('No news data — add NewsAPI key to configs/config.local.yaml')

News corpus: 344 articles
Date range : 2026-03-06 16:33:54 → 2026-03-29 13:28:13

Articles per asset:
symbol
BTC-USD     95
GC=F        83
EURUSD=X    80
ETH-USD     69
KES=X       17

Sample headlines:
  [GC=F] We Found Nearly 100 of the Best Deals From Amazon's Big Spring Sale
  [ETH-USD] MrBeast is drawing political heat after buying a banking app aimed at teens and 
  [GC=F] DAVID BLACKMON: Competition, Not Monopoly Control, The Answer To Grid Reliabilit
  [EURUSD=X] Swiss National Bank keeps rates at zero, eyes Iran war
  [ETH-USD] CZR Exchange Launches CZR DEX, Expanding Into High-Performance On-Chain Trading


In [12]:
macro = load_macro()
if macro is not None and not macro.empty:
    print(f'FRED macro: {macro.shape[0]:,} rows × {macro.shape[1]} series')
    print(f'Columns   : {list(macro.columns)}')
    print(f'Date range: {macro.index[0].date()} → {macro.index[-1].date()}')

    fig, axes = plt.subplots(len(macro.columns), 1,
                             figsize=(13, 2.5 * len(macro.columns)), sharex=True)
    if len(macro.columns) == 1: axes = [axes]

    series_colours = ['#378ADD','#1D9E75','#BA7517','#D85A30','#7F77DD','#D4AF37']
    series_labels  = {
        'CPIAUCSL': 'US CPI (inflation)',
        'DFF':      'Fed Funds Rate (%)',
        'T10Y2Y':   '10Y-2Y Spread (%)',
        'DEXUSEU':  'USD/EUR Rate',
        'DEXCHUS':  'USD/CNY Rate',
        'GOLDAMGBD228NLBM': 'Gold London Fix',
    }
    for ax, col, col_colour in zip(axes, macro.columns, series_colours):
        series = macro[col].dropna()
        ax.plot(series.index, series.values, color=col_colour, linewidth=0.9)
        ax.fill_between(series.index, series.values, series.values.min(),
                        alpha=0.08, color=col_colour)
        ax.set_ylabel(series_labels.get(col, col), fontsize=8)

    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.suptitle('FRED Macro Indicators', fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig('logs/06_macro_fred.png', dpi=130, bbox_inches='tight')
    plt.show()
else:
    print('No macro data — add FRED API key to configs/config.local.yaml')

FRED macro: 2,192 rows × 4 series
Columns   : ['CPIAUCSL', 'DFF', 'T10Y2Y', 'DEXCHUS']
Date range: 2019-01-01 → 2024-12-31


In [13]:
#  All assets feature summary

all_symbols = [a['symbol'] for a in
    config['assets']['forex'] + config['assets']['crypto'] + config['assets']['commodities']]

summary_rows = []
for sym in all_symbols:
    f = load_features(sym)
    if f is not None:
        fc = get_feature_columns(f)
        summary_rows.append({
            'Symbol':    sym,
            'Name':      LABELS.get(sym, sym),
            'Rows':      len(f),
            'Features':  len(fc),
            'Start':     str(f.index[0].date()),
            'End':       str(f.index[-1].date()),
            'Ann.Vol%':  round(f['log_return'].std() * (252**0.5) * 100, 1),
            'UpDays%':   round((f['return_1d'] > 0).mean() * 100, 1),
        })

summary = pd.DataFrame(summary_rows)
print('Phase 1 Feature Summary')
print('=' * 70)
print(summary.to_string(index=False))

Phase 1 Feature Summary
  Symbol     Name  Rows  Features      Start        End  Ann.Vol%  UpDays%
EURUSD=X  EUR/USD  1515        23 2019-03-11 2024-12-30       7.1     49.2
   KES=X  USD/KES  1515        23 2019-03-11 2024-12-30      10.3     54.7
 BTC-USD  Bitcoin  2142        23 2019-02-19 2024-12-30      54.7     51.4
 ETH-USD Ethereum  2142        23 2019-02-19 2024-12-30      69.2     51.9
    GC=F     Gold  1460        23 2019-03-14 2024-12-30      15.7     54.7
